# Validação geográfica externa — Projeto de mestrado

**Objetivo:** aplicar os classificadores treinados exclusivamente na Area1 a regiões de Novo Hamburgo e São Leopoldo.

**Preservação:** este notebook não altera a branch `main`, o README nem os resultados históricos. As imagens de vários gigabytes não são adicionadas ao GitHub.

**Regra científica:** mapas de previsão **não** são F1-macro. Somente rótulos locais independentes permitem pontuar o desempenho externo.

In [ ]:
import subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJETO=Path('/content/mestrado-arvores-urbanas-externo')
if not PROJETO.exists():
    subprocess.run(['git','clone','--depth=1','--branch','feature/validacao-geografica-externa',
                    'https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git',str(PROJETO)],check=True)
SCRIPTS=PROJETO/'experimentos/validacao_externa_regioes'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(SCRIPTS/'requirements.txt')],check=True)
print('Código externo carregado de:',SCRIPTS)

## 1. Selecionar a região

A pasta `Imagens do Mestrado` deve estar disponível no Google Drive montado nesta sessão, pelo mesmo usuário ou por um **atalho em Meu Drive**. Um link de outra conta não é automaticamente montado no Colab.

**Para economizar espaço, processe uma região por vez.** Os arquivos compactados têm aproximadamente 1,5–3,1 GB cada, e a extração requer espaço adicional.

In [ ]:
REGIAO='centro_novo_hamburgo'  # outras: lomba_grande_novo_hamburgo, cristo_rei_sao_leopoldo, sagrada_familia_caxias
ARQUIVOS={
  'centro_novo_hamburgo':'CENTRO_NOVO HAMBURGO.zip',
  'lomba_grande_novo_hamburgo':'LOMBA_GRANDE_NOVO_HAMBURGO.zip',
  'cristo_rei_sao_leopoldo':'CRISTO_REI_SÃO_LEOPOLDO.rar',
  'sagrada_familia_caxias':'SAGRADA_FAMÍLIA_CAXIAS DO SUL.zip',
}
PASTA_DRIVE=Path('/content/drive/MyDrive/Imagens do Mestrado')
BASE_AREA1=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1')
CSV_AREA1=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/v2_progressivo_256/rotulos_sem_duplicacao.csv')
GPKG_GEOMETRIAS_AREA1=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/regularizacao_2026_11_03/entradas/E3c_geometrias_rotuladas_812.gpkg')
SAIDA_DRIVE=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/validacao_externa_regioes')
LOCAL=Path('/content/validacao_externa_local')
SAIDA_DRIVE.mkdir(parents=True,exist_ok=True)
PACOTE_ORIGEM=PASTA_DRIVE/ARQUIVOS[REGIAO]
assert PACOTE_ORIGEM.is_file(), f'Arquivo não encontrado no Drive: {PACOTE_ORIGEM}. Ajuste o caminho ou crie atalho.'
print('Região:',REGIAO,'\nPacote:',PACOTE_ORIGEM,'\nBytes:',PACOTE_ORIGEM.stat().st_size)

## 2. Inventariar ZIP/RAR e extrair os arquivos

Verifique se os rasters contêm NIR. O pipeline principal **exige quatro bandas** para reproduzir os atributos da Area1. Se o arquivo possuir somente RGB, o código para sem inventar NDVI.

In [ ]:
from shutil import copy2
LOCAL.mkdir(exist_ok=True)
DIR_ARQUIVO=LOCAL/'compactados'
DIR_ARQUIVO.mkdir(exist_ok=True)
ARQUIVO_LOCAL=DIR_ARQUIVO/PACOTE_ORIGEM.name
if not ARQUIVO_LOCAL.is_file() or ARQUIVO_LOCAL.stat().st_size != PACOTE_ORIGEM.stat().st_size:
    copy2(PACOTE_ORIGEM,ARQUIVO_LOCAL)
DIR_DADOS=LOCAL/'extraidos'
AUDITORIA=SAIDA_DRIVE/'auditorias'/REGIAO
command=[sys.executable,str(SCRIPTS/'auditar_pacotes.py'),
         '--arquivos',str(DIR_ARQUIVO),'--extraidos',str(DIR_DADOS),
         '--extrair','--out',str(AUDITORIA)]
if (BASE_AREA1/'AOI_RGB_NEW.tif').is_file():
    command+=['--rgb-area1',str(BASE_AREA1/'AOI_RGB_NEW.tif')]
subprocess.run(command,check=True)
print('Auditoria:',AUDITORIA/'auditoria_pacotes.json')

In [ ]:
import json
audit=json.loads((AUDITORIA/'auditoria_pacotes.json').read_text())
for item in audit['arquivos_inspecionados']:
    if item['tipo'].startswith('raster') or item['tipo'].startswith('vetor'):
        print(item['tipo'],item['caminho'],'bandas:',item.get('bandas'),
              'CRS:',item.get('crs'),'dimensões:',item.get('largura'),item.get('altura'))

## 3. Identificar explicitamente os arquivos RGB e multiespectral

Depois de ler o relatório da auditoria, preencha os caminhos abaixo. Não selecione automaticamente um TIFF apenas pelo nome: verifique ordem das bandas, resolução e CRS.

Se houver **um único TIFF com pelo menos quatro bandas RGB+NIR**, ele pode ser usado como RGB e multiespectral, **desde que a ordem das bandas seja confirmada**.

Se faltar NIR, suspenda a comparação quantitativa e documente a limitação.

In [ ]:
# PREENCHER com caminhos COMPLETOS que foram mostrados na auditoria:
RGB_TIF = None
MULTI_TIF = None
AOI_VETOR = None       # opcional, um .shp, .gpkg ou .geojson da região
if RGB_TIF is None or MULTI_TIF is None:
    raise RuntimeError('Informe RGB_TIF e MULTI_TIF após confirmar as bandas do pacote.')
RGB_TIF, MULTI_TIF = Path(RGB_TIF), Path(MULTI_TIF)
assert RGB_TIF.is_file() and MULTI_TIF.is_file()
print('RGB:',RGB_TIF,'\nRGB + NIR:',MULTI_TIF)

## 4. Ajustar os oito classificadores na Area1

Os hiperparâmetros são selecionados por **validação espacial interna**, usando somente dados da Area1. Os novos municípios **não** participam do treinamento nem da seleção.

Se `CSV_AREA1` não estiver presente, o notebook tentará **reconstruir a base rotulada da Area1 diretamente dos rasters e referências originais**, com o mesmo Quickshift dos quadrantes, interrompendo caso a contagem não corresponda aos **505 segmentos** documentados. Esse procedimento produz um novo artefato auditável, sem substituir os arquivos históricos.

O embargo geométrico requer o GeoPackage de geometrias da Area1. Um modo alternativo de exclusão por centroides é explicitamente diferente.

In [ ]:
if not CSV_AREA1.is_file():
    print('CSV Area1 não encontrado. Reconstruindo a base rotulada com o Quickshift original.')
    RECONSTRUIDA=SAIDA_DRIVE/'base_area1_reconstituida'
    cmd=[sys.executable,str(SCRIPTS/'extrair_base_area1.py'),
         '--area1',str(BASE_AREA1),'--out',str(RECONSTRUIDA)]
    if (RECONSTRUIDA/'assinatura_reconstrucao.json').exists():
        cmd.append('--retomar')
    subprocess.run(cmd,check=True)
    CSV_AREA1=RECONSTRUIDA/'rotulos_sem_duplicacao.csv'
assert CSV_AREA1.is_file(), ('Base rotulada de 505 segmentos não encontrada: '+str(CSV_AREA1))
MODEL_DIR=SAIDA_DRIVE/'modelos_area1'
if not (MODEL_DIR/'manifesto_modelos.json').is_file():
    cmd=[sys.executable,str(SCRIPTS/'treinar_area1.py'),
         '--area1-csv',str(CSV_AREA1),'--out',str(MODEL_DIR)]
    if GPKG_GEOMETRIAS_AREA1.is_file():
        cmd+=['--geometrias',str(GPKG_GEOMETRIAS_AREA1)]
    else:
        print('AVISO: sem GeoPackage, embargo apenas por centroides (protocolo explicitamente diferente).')
        cmd+=['--permitir-embargo-centroides']
    subprocess.run(cmd,check=True)
print('Modelos:',MODEL_DIR)

## 5. Aplicar os modelos à região externa

O processamento percorre os rasters em janelas de até **256 × 256 pixels**. A quantidade de quadrantes é calculada automaticamente para cada raster.

Os resultados são mapas GeoTIFF, por classificador, e atributos por segmento. Use `--retomar` se uma execução tiver sido interrompida e os arquivos de entrada/modelos forem exatamente os mesmos.

In [ ]:
# Para testar rapidamente, ajuste PRIMEIRO para 3. Depois rode novamente com LIMITE_QUADRANTES=None e retomar=True.
LIMITE_QUADRANTES=None  # 3 para teste de fumaça; None para completar todos os quadrantes
DIR_PREDICOES=SAIDA_DRIVE/'predicoes'/REGIAO
args=[sys.executable,str(SCRIPTS/'aplicar_regiao.py'),
      '--regiao',REGIAO,'--rgb',str(RGB_TIF),'--multi',str(MULTI_TIF),
      '--modelos',str(MODEL_DIR),'--out',str(DIR_PREDICOES)]
if AOI_VETOR:
    args+=['--aoi',str(AOI_VETOR)]
if (DIR_PREDICOES/'configuracao_execucao.json').exists():
    args+=['--retomar']
if LIMITE_QUADRANTES is not None:
    args+=['--limite-quadrantes',str(LIMITE_QUADRANTES)]
# Alinhamento é intervenção metodológica explícita. Só habilite após auditar os rasters:
# args += ['--alinhar-multiespectral']
subprocess.run(args,check=True)
print(json.loads((DIR_PREDICOES/'manifesto_execucao.json').read_text())['estado'])

## 6. Avaliar F1-macro somente com referências independentes

A classificação pode produzir mapas **sem** rótulos locais, mas isso não é validação externa quantitativa. Para medir F1-macro, indique arquivos GeoPackage/SHAPEFILE com pontos de árvores e de gramíneas **das regiões novas**, nunca rótulos usados no ajuste dos modelos.

In [ ]:
REF_ARVORES=None   # exemplo: '/content/.../Arvores.gpkg' (rótulos independentes da nova região)
REF_GRAMA=None     # exemplo: '/content/.../Grama.gpkg'
if REF_ARVORES and REF_GRAMA:
    DIR_METRICAS=SAIDA_DRIVE/'metricas'/REGIAO
    cmd=[sys.executable,str(SCRIPTS/'avaliar_regiao.py'),'--regiao',REGIAO,
         '--predicoes',str(DIR_PREDICOES),'--arvores',str(REF_ARVORES),
         '--grama',str(REF_GRAMA),'--out',str(DIR_METRICAS)]
    subprocess.run(cmd,check=True)
    print('Resultados externos:',DIR_METRICAS/'metricas_externas.csv')
else:
    print('Sem rótulos locais independentes: mapas disponíveis, mas F1 externo não calculado.')

## 7. Comparar os resultados entre regiões

Após processar e validar as três regiões, reúna os arquivos `metricas_externas.csv` de cada uma. Se não houver rótulos, registre apenas mapas e limitações.

**Não copie valores históricos da Area1 para simular resultados externos.** Não edite o README até haver resultados executados e verificados.

In [ ]:
import pandas as pd
csvs=list((SAIDA_DRIVE/'metricas').glob('*/metricas_externas.csv')) if (SAIDA_DRIVE/'metricas').exists() else []
if csvs:
    comparacao=pd.concat([pd.read_csv(f) for f in csvs],ignore_index=True)
    display(comparacao.sort_values(['regiao','f1_macro'],ascending=[True,False]))
else:
    print('Ainda não há métricas externas verificadas. Este estado é esperado sem rótulos locais.')